# DesignLab v9.0 prospective Pilot B

I use this single-runner-cell notebook to verify my uploaded bundle, load my Colab Secret, and make four OpenRouter engineering connectivity probes. I run the 5-item Pilot B pipeline only if all four model slots are available. My estimated-spend cap is USD 0.25. This is prospective pilot work, not confirmatory-study data.

I add `OPENROUTER_API_KEY` as a Colab Secret. This cell loads it from `google.colab.userdata` into the process environment and never prints it. I upload `v9_0_colab_pilot_b_bundle.zip` when prompted. A T4 runtime works; these HTTP API calls do not use the GPU.

When I run this cell, it makes actual OpenRouter API calls. Preparing the notebook and bundle makes no provider requests.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

from google.colab import files

EXPECTED_POOL_SHA256 = "128ff78bb6beb1640fa51897ab97dbb9d07dc56de276973d05a48dda524b53d3"
NUM_ITEMS = 5  # Set to 50 for Pilot A or 260 for a separately authorized confirmatory run.
BUNDLE_PATH = Path("/content/v9_0_colab_pilot_b_bundle.zip")
EXTRACT_ROOT = Path("/content/v9_0_colab_pilot_b_bundle")

if not BUNDLE_PATH.is_file():
    print("Upload v9_0_colab_pilot_b_bundle.zip when the file picker opens.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise RuntimeError("Upload exactly one .zip bundle.")
    BUNDLE_PATH = Path("/content") / zip_names[0]

if not EXTRACT_ROOT.exists():
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(BUNDLE_PATH) as zf:
        for info in zf.infolist():
            member = Path(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError("Unsafe path found in bundle archive.")
        zf.extractall(EXTRACT_ROOT)

RESEARCH_DIR = EXTRACT_ROOT / "research" / "v9_0_prospective"
POOL_PATH = EXTRACT_ROOT / "items_pool_v1.json"
PING_SCRIPT = RESEARCH_DIR / "api_ping_check_v9.py"
PILOT_SCRIPT = RESEARCH_DIR / "pilot_b_v9.py"
RATE_CARD = RESEARCH_DIR / "model_rates_v9.json"

required_files = [POOL_PATH, PING_SCRIPT, PILOT_SCRIPT, RATE_CARD,
                  RESEARCH_DIR / "MANIFEST_SHA256.txt"]
missing_files = [str(p.name) for p in required_files if not p.is_file()]
if missing_files:
    raise FileNotFoundError("Bundle is missing required files: " + ", ".join(missing_files))

pool_bytes = POOL_PATH.read_bytes()
pool_hash = hashlib.sha256(pool_bytes).hexdigest()
if pool_hash != EXPECTED_POOL_SHA256:
    raise RuntimeError("Frozen item-pool SHA-256 mismatch; no keys were loaded and no calls were sent.")
pool_data = json.loads(pool_bytes.decode("utf-8"))
required_fields = {"item_id", "category", "prompt_text"}
if (not isinstance(pool_data, list) or len(pool_data) != 260 or
    any(not isinstance(item, dict) or set(item) != required_fields for item in pool_data)):
    raise RuntimeError("Frozen item-pool schema check failed; no keys were loaded and no calls were sent.")

manifest_path = RESEARCH_DIR / "MANIFEST_SHA256.txt"
manifest_rows = []
for line in manifest_path.read_text(encoding="ascii").splitlines():
    expected, relative = line.split("  ", 1)
    target = RESEARCH_DIR / relative
    actual = hashlib.sha256(target.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError("Source manifest mismatch for " + relative)
    manifest_rows.append(relative)

if len({str(item.get("item_id")) for item in pool_data}) != 260:
    raise RuntimeError("Frozen item-pool IDs are not unique.")
if any(not isinstance(item.get("prompt_text"), str) or not item["prompt_text"].strip() for item in pool_data):
    raise RuntimeError("Frozen item-pool contains an empty prompt.")

RUNS_ROOT = Path("/content/v9_0_colab_pilot_b_runs")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")
RUN_DIR = RUNS_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("Bundle files and SHA-256 manifest verified.")
print(f"Pool verified: {len(pool_data)} items; SHA-256 {pool_hash}; {len(manifest_rows)} source files.")
print(f"Run output directory: {RUN_DIR}")

from google.colab import userdata

SECRET_NAMES = ["OPENROUTER_API_KEY"]
missing_secrets = []
for name in SECRET_NAMES:
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if isinstance(value, str) and value.strip():
        os.environ[name] = value
    else:
        missing_secrets.append(name)

if missing_secrets:
    raise RuntimeError("Add these Colab Secrets before continuing: " + ", ".join(missing_secrets))

# Keep references only in memory for a post-run leak scan; never display these values.
SECRET_VALUES = [os.environ[name] for name in SECRET_NAMES]
print("Loaded the OpenRouter Colab Secret into the environment. Secret value is suppressed.")

PING_PATH = RUN_DIR / "api_ping_check_v9.json"
PING_LOG_PATH = RUN_DIR / "api_ping_check_v9.log"
ping_proc = subprocess.run(
    [sys.executable, str(PING_SCRIPT), "--execute", "--output", str(PING_PATH)],
    cwd=str(RESEARCH_DIR), capture_output=True, text=True, check=False,
)
PING_LOG_PATH.write_text(
    "exit_code=" + str(ping_proc.returncode) + "\n" + (ping_proc.stdout or ""),
    encoding="utf-8",
)

ping_report = json.loads(PING_PATH.read_text(encoding="utf-8")) if PING_PATH.is_file() else {}
ping_rows = ping_report.get("results", [])
for row in ping_rows:
    state = "AVAILABLE" if row.get("available") else "REJECTED"
    blockers = ",".join(row.get("availability_blockers", [])) or "none"
    print(f"{state}: {row.get('provider')} / {row.get('model_id')} | HTTP {row.get('http_status')} | blockers={blockers}")

PING_OK = (
    ping_proc.returncode == 0
    and len(ping_rows) == 4
    and all(row.get("available") is True for row in ping_rows)
)
print(f"Connectivity gate: {'PASS' if PING_OK else 'FAIL'}; report: {PING_PATH.name}")


PILOT_PATH = RUN_DIR / "pilot_b_v9_result.json"
PILOT_LOG_PATH = RUN_DIR / "pilot_b_v9_run.log"
PILOT_RAN = False
pilot_report = {}

if PING_OK:
    pilot_proc = subprocess.run(
        [
            sys.executable, str(PILOT_SCRIPT),
            "--execute",
            "--items", str(POOL_PATH),
            "--num-items", str(NUM_ITEMS),
            "--rates", str(RATE_CARD),
            "--budget-usd", "0.25",
            "--output", str(PILOT_PATH),
        ],
        cwd=str(RESEARCH_DIR), capture_output=True, text=True, check=False,
    )
    PILOT_RAN = True
    PILOT_LOG_PATH.write_text(
        "exit_code=" + str(pilot_proc.returncode) + "\n" + (pilot_proc.stdout or ""),
        encoding="utf-8",
    )
    if PILOT_PATH.is_file():
        pilot_report = json.loads(PILOT_PATH.read_text(encoding="utf-8"))
    print(pilot_proc.stdout.strip() or "Pilot process produced no standard output.")
    if pilot_proc.returncode != 0:
        print("Pilot ended with a nonzero status; its checkpoint report will be included in the result bundle when present.")
else:
    skip = {
        "run_classification": "prospective_pilot_b_skipped",
        "confirmatory_data_collected": False,
        "reason": "connectivity gate did not pass for all four providers",
        "item_pool_sha256": EXPECTED_POOL_SHA256,
        "pilot_calls_started": 0,
    }
    (RUN_DIR / "pilot_b_v9_status.json").write_text(json.dumps(skip, indent=2) + "\n", encoding="utf-8")
    (RUN_DIR / "pilot_b_v9_run.log").write_text("Pilot B skipped because the connectivity gate failed.\n", encoding="utf-8")
    print("Pilot B skipped. No Pilot B requests were sent.")

# Derive a token/spend report and a JSONL call log from the redacted checkpoint report.
calls = pilot_report.get("calls", []) if pilot_report else []
calls_path = RUN_DIR / "pilot_b_v9_calls.jsonl"
if pilot_report:
    calls_path.write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in calls),
        encoding="utf-8",
    )

token_spend = {
    "run_classification": pilot_report.get("run_classification", "prospective_pilot_b_skipped"),
    "confirmatory_data_collected": pilot_report.get("confirmatory_data_collected", False),
    "num_items_requested": pilot_report.get("num_items_requested", NUM_ITEMS),
    "num_items_selected": pilot_report.get("num_items_selected", 0),
    "item_pool_sha256": EXPECTED_POOL_SHA256,
    "budget_cap_usd": pilot_report.get("budget_cap_usd", 0.25),
    "budget_reserved_usd": pilot_report.get("budget_reserved_usd", 0.0),
    "total_estimated_usd": pilot_report.get("total_estimated_usd", 0.0),
    "http_calls_started": pilot_report.get("http_calls_started", 0),
    "http_responses_received": pilot_report.get("http_responses_received", 0),
    "logical_call_records": pilot_report.get("logical_call_records", 0),
    "maximum_logical_calls_with_shared_scrub_fallback": pilot_report.get("maximum_logical_calls_with_shared_scrub_fallback", 0),
    "maximum_http_attempts_with_transient_retries": pilot_report.get("maximum_http_attempts_with_transient_retries", 0),
    "halt_reason": pilot_report.get("halt_reason"),
    "per_model_summary": pilot_report.get("per_model_summary", []),
    "per_billing_model_summary": pilot_report.get("per_billing_model_summary", []),
}
(RUN_DIR / "pilot_b_v9_token_spend_report.json").write_text(
    json.dumps(token_spend, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)

# Manifest all generated report/log files. It excludes itself and the result ZIP.
manifest_out = RUN_DIR / "SHA256SUMS.txt"
artifact_paths = sorted(
    p for p in RUN_DIR.iterdir()
    if p.is_file() and p.name not in {manifest_out.name, "v9_0_pilot_b_results.zip"}
)
manifest_out.write_text(
    "".join(f"{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.name}\n" for p in artifact_paths),
    encoding="ascii",
)

RESULTS_ZIP = Path("/content/v9_0_pilot_b_results.zip")
if RESULTS_ZIP.exists():
    RESULTS_ZIP.unlink()
with zipfile.ZipFile(RESULTS_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(p for p in RUN_DIR.iterdir() if p.is_file()):
        zf.write(path, arcname=path.name)

secret_leaks = []
pattern_leaks = []
key_patterns = [
    re.compile(rb"AIza[0-9A-Za-z_-]{20,}"),
    re.compile(rb"sk-[0-9A-Za-z_-]{20,}"),
    re.compile(rb"gsk_[0-9A-Za-z_-]{20,}"),
    re.compile(rb"\?key="),
    re.compile(rb"(?i:Bearer)\s+[A-Za-z0-9._~+/-]{16,}={0,}"),
]
with zipfile.ZipFile(RESULTS_ZIP, "r") as zf:
    for name in zf.namelist():
        payload = zf.read(name)
        for secret_name, secret_value in zip(SECRET_NAMES, SECRET_VALUES):
            if secret_value.encode("utf-8") in payload:
                secret_leaks.append(secret_name)
        if any(pattern.search(payload) for pattern in key_patterns):
            pattern_leaks.append(name)
if secret_leaks or pattern_leaks:
    raise RuntimeError(
        "Credential scan failed. Secret names affected: "
        + ",".join(sorted(set(secret_leaks)))
        + "; pattern matches: " + ",".join(pattern_leaks)
    )

results_sha256 = hashlib.sha256(RESULTS_ZIP.read_bytes()).hexdigest()
print(f"Result archive created: {RESULTS_ZIP.name}")
print(f"Artifacts included: {len(artifact_paths) + 1} (including SHA256SUMS.txt)")
print("API-key scan: zero matches in archive members.")
print("SHA-256: " + results_sha256)
files.download(str(RESULTS_ZIP))